# Respiration Analysis – RI1 Juvenile Interaction

## 1. Imports & Paths
## 2. Load H5 Files
## 3. Load & Preprocess Respiration
## 4. Session QC & Exclusions
## 5. Respiration Rate Extraction
## 6. Grouping (ChR vs GFP)
## 7. Summary Plots & Tables


In [78]:
import pandas as pd
import numpy as np
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import neurokit2 as nk
import spikeinterface.extractors as se
import matplotlib.pyplot as plt
import h5py
import glob
import glob
import os
import os
import h5py
import numpy as np
from scipy.signal import butter, filtfilt, resample_poly
import neurokit2 as nk


In [79]:
H5_DIRS = {
    "day1": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day1\h5_outputs",
    "day2": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day2\h5_outputs",
}


preprocessing signal > 

In [84]:
for day, d in H5_DIRS.items():
    print(day, len(glob.glob(os.path.join(d, "*.h5"))))


day1 25
day2 23


In [85]:
sessions_df = pd.DataFrame([
    {
        "session": sess,
        "day": data["day"]
    }
    for sess, data in resp_all.items()
])

sessions_df.sort_values(["day", "session"]).reset_index(drop=True)


,session,day
0,"(1_1_RI1_f1_d_on_20260111_164638, day1)",day1
1,"(1_2_RI1_f2_f_off_20260109_165330, day1)",day1
2,"(1_3_RI1_f1_c_off_20260110_130254, day1)",day1
3,"(2_1_RI1_f2_h_on_20260110_145355, day1)",day1
4,"(2_2_RI1_f1_b_off_20260109_150435, day1)",day1
5,"(2_3_RI1_f2_f_on_20260110_132734, day1)",day1
6,"(3_1_RI1_f1_b_on_20260111_154618, day1)",day1
7,"(3_2_RI1_f1_a_off_20260114_161949, day1)",day1
8,"(3_2_RI1_f2_g_off_20260110_122606, day1)",day1
9,"(3_3_RI1_f1_c_on_20260111_162426, day1)",day1


In [80]:
def load_clean_resp_signal_ecu(h5_file, target_rate=100, fs=20000):
    """
    Loads, filters, downsamples respiration from ECU-style .h5.
    Preserves original preprocessing logic.
    
    Returns:
        rsp_cleaned : np.ndarray
        time_vector : np.ndarray
        fs_out      : float (target_rate)
        metadata    : dict
    """

    try:
        with h5py.File(h5_file, 'r') as f:

            # -----------------------------
            # Load respiration (ECU Ain1)
            # -----------------------------
            analog_keys = list(f['analog'].keys())
            resp_keys = [k for k in analog_keys if "Ain1" in k]
            if len(resp_keys) == 0:
                raise RuntimeError("No ECU Ain1 channel found")

            resp_key = resp_keys[0]
            resp = f['analog'][resp_key][()]['voltage'].astype(float)

            # -----------------------------
            # Metadata (keep for compatibility)
            # -----------------------------
            metadata = {
                "fs_original": fs,
                "source": "ECU_Ain1",
                "file": os.path.basename(h5_file)
            }

    except Exception as e:
        print(f" Error loading {os.path.basename(h5_file)}: {e}")
        return None, None, None, None

    # -----------------------------
    # PRE-CLEAN (CRITICAL)
    # -----------------------------
    # Remove NaNs / infs
    bad = ~np.isfinite(resp)
    if np.any(bad):
        resp[bad] = np.nanmedian(resp)

    # Remove DC offset
    resp = resp - np.median(resp)

    # Clip extreme artifacts (opto / motion)
    mad = np.median(np.abs(resp))
    if mad > 0:
        resp = np.clip(resp, -10 * mad, 10 * mad)

    # -----------------------------
    # Low-pass before downsampling
    # -----------------------------
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # -----------------------------
    # Downsample
    # -----------------------------
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # -----------------------------
    # Bandpass (NeuroKit)
    # -----------------------------
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # -----------------------------
    # Time vector
    # -----------------------------
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate, metadata

In [81]:
resp_all = {}

for day, h5_dir in H5_DIRS.items():
    h5_files = sorted(glob.glob(os.path.join(h5_dir, "*.h5")))

    for h5_file in h5_files:
        sess = os.path.basename(h5_file).replace(".h5", "")

        rsp, t, fs_out, meta = load_clean_resp_signal_ecu(h5_file)

        if rsp is None:
            continue

        resp_all[(sess, day)] = {
            "resp": rsp,
            "time": t,
            "fs": fs_out,
            "meta": meta,
            "day": day
        }


In [82]:
def animal_id_from_sess(sess):
    return "_".join(sess.split("_")[:2])


In [83]:
included_sessions = [
    sess for sess in resp_all
    if animal_id_from_sess(sess) not in excluded_animals
]


AttributeError: 'tuple' object has no attribute 'split'

In [ ]:
results = []

for sess in included_sessions:
    data = resp_all[sess]

    rsp = data["resp"]
    t   = data["time"]
    fs  = data["fs"]
    day = data["day"]

    cage = int(sess.split("_")[0])
    animal_id = animal_id_from_sess(sess)
    sex = "F" if cage <= 4 else "M"

    group, virus = assign_group_and_virus(sess)
    laser = "on" if "_on_" in sess else "off"

    rates = {
        "baseline": get_sniff_respiratory_rate(rsp, t, 0, 60, fs),
        "social":   get_sniff_respiratory_rate(rsp, t, 60, 240, fs)
    }

    for epoch, rate in rates.items():
        results.append({
            "session": sess,
            "animal_id": animal_id,
            "day": day,
            "cage": cage,
            "sex": sex,
            "group": group,
            "virus": virus,
            "laser": laser,
            "epoch": epoch,
            "resp_rate_hz": rate
        })


In [ ]:
import pandas as pd
df = pd.DataFrame(results)


In [ ]:
df.groupby("animal_id")["day"].nunique()
df.groupby("animal_id")["laser"].nunique()
df.groupby(["animal_id", "day"]).size()


KeyError: 'animal_id'

In [ ]:
import numpy as np
from scipy.signal import find_peaks


def get_sniff_respiratory_rate(
    signal,
    time,
    sniff_start,
    sniff_end,
    sampling_rate=100,
    max_rate_hz=12
):
    """
    Compute average respiration/sniff rate using inter-breath intervals (IBI).
    
    Returns:
        avg_rate_hz (float) or np.nan
    """

    # -----------------------------
    # Mask window
    # -----------------------------
    sniff_mask = (time >= sniff_start) & (time < sniff_end)

    if sniff_mask.sum() < sampling_rate:  # <1 s of data
        return np.nan

    signal_sniff = signal[sniff_mask]
    time_sniff = time[sniff_mask]

    # -----------------------------
    # Pre-clean for peak detection
    # -----------------------------
    signal_sniff = signal_sniff - np.mean(signal_sniff)

    # -----------------------------
    # Peak detection
    # -----------------------------
    min_ibi = 1.0 / max_rate_hz
    min_distance = int(min_ibi * sampling_rate)

    peaks, _ = find_peaks(
        signal_sniff,
        distance=min_distance
    )

    if len(peaks) < 2:
        return np.nan

    # -----------------------------
    # IBI and rate
    # -----------------------------
    peak_times = time_sniff[peaks]
    ibi = np.diff(peak_times)          # seconds
    inst_rate = 1.0 / ibi              # Hz

    return np.mean(inst_rate)


In [ ]:
excluded_animals = {
    "2_3",
    "3_1",
    "3_2",
    "5_1",
    "5_2",
    "5_3",
    "6_2",
    "6_3",
    "8_1",
}

In [ ]:
def assign_sex(sess):
    cage = int(sess.split("_")[0])
    return "F" if cage <= 4 else "M"


In [ ]:
def assign_laser(sess):
    return "on" if "_on_" in sess else "off"


In [ ]:
def assign_group_and_virus(sess):
    cage_num = int(sess.split("_")[0])
    if cage_num % 2 == 1:
        return "A", "ChR"
    else:
        return "B", "GFP"

In [ ]:
results = []

for sess, data in resp_all.items():

    if sess not in included_sessions:
        continue

    rsp = data["resp"]
    t   = data["time"]
    fs  = data["fs"]

    group, virus = assign_group_and_virus(sess)
    laser = assign_laser(sess)
    sex   = assign_sex(sess)

    baseline_rate = get_sniff_respiratory_rate(
        rsp, t, 0, 60, sampling_rate=fs
    )

    social_rate = get_sniff_respiratory_rate(
        rsp, t, 60, 240, sampling_rate=fs
    )

    results.append({
        "session": sess,
        "animal_id": sess.split("_RI1")[0],
        "cage": int(sess.split("_")[0]),
        "sex": sex,
        "group": group,
        "virus": virus,
        "laser": laser,
        "baseline_rate_hz": baseline_rate,
        "social_rate_hz": social_rate,
        "delta_rate_hz": social_rate - baseline_rate
    })


In [ ]:
import pandas as pd
df = pd.DataFrame(results)


In [ ]:
df.head()


""


In [ ]:
def is_excluded(sess, excluded_animals):
    # assumes session names start with animal ID like "3_2_..."
    animal_id = sess.split("_RI1")[0]
    return animal_id in excluded_animals

included_sessions = [
    sess for sess in resp_all.keys()
    if not is_excluded(sess, excluded_animals)
]

print("Included sessions:")
for s in included_sessions:
    print(" ", s)


Included sessions:


In [ ]:
results = []

for sess in included_sessions:
    data = resp_all[sess]
    rsp = data["resp"]
    t   = data["time"]
    fs  = data["fs"]

    group, virus = assign_group_and_virus(sess)
    laser = "on" if "_on_" in sess else "off"

    rates = {
        "baseline": get_sniff_respiratory_rate(
            rsp, t, 0, 60, sampling_rate=fs
        ),
        "social": get_sniff_respiratory_rate(
            rsp, t, 60, 240, sampling_rate=fs
        )
    }

    for epoch, rate in rates.items():
        results.append({
            "session": sess,
            "animal_id": sess.split("_RI1")[0],
            "cage": int(sess.split("_")[0]),
            "group": group,
            "virus": virus,
            "laser": laser,
            "epoch": epoch,
            "resp_rate_hz": rate
        })


In [ ]:
list(resp_all.keys())[15]


'5_3_RI1_m1_k_off_20260109_183135'

In [ ]:
type(next(iter(resp_all.keys())))


str

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df.head()


,session,animal_id,cage,group,virus,laser,epoch,resp_rate_hz
0,1_1_RI1_f1_d_on_20260111_164638,1_1,1,A,ChR,on,baseline,9.263738
1,1_1_RI1_f1_d_on_20260111_164638,1_1,1,A,ChR,on,social,9.152318
2,1_2_RI1_f2_f_off_20260109_165330,1_2,1,A,ChR,off,baseline,9.683383
3,1_2_RI1_f2_f_off_20260109_165330,1_2,1,A,ChR,off,social,10.288113
4,1_3_RI1_f1_c_off_20260110_130254,1_3,1,A,ChR,off,baseline,9.113699


In [ ]:
df.head()


,session,animal_id,cage,group,virus,laser,epoch,resp_rate_hz
0,1_1_RI1_f1_d_on_20260111_164638,1_1,1,A,ChR,on,baseline,9.263738
1,1_1_RI1_f1_d_on_20260111_164638,1_1,1,A,ChR,on,social,9.152318
2,1_2_RI1_f2_f_off_20260109_165330,1_2,1,A,ChR,off,baseline,9.683383
3,1_2_RI1_f2_f_off_20260109_165330,1_2,1,A,ChR,off,social,10.288113
4,1_3_RI1_f1_c_off_20260110_130254,1_3,1,A,ChR,off,baseline,9.113699


In [ ]:
df.columns.tolist()


['session',
 'animal_id',
 'cage',
 'group',
 'virus',
 'laser',
 'epoch',
 'resp_rate_hz']

In [ ]:
sex_colors = {
    "M": "#1f77b4",   # blue
    "F": "#e377c2"    # pink
}
